# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs) + specificity gate

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 23** you build `fp.Design` **binder** objects from the pools, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per sequence designer** (LigandMPNN vs ProteinMPNN) so the head-to-head is fair (D3 part 1). On top of
the shared confidence layers we add a **specificity gate** (prefer the intended motif over a scrambled
one) — because for protein–NA binders, *confidence is not specificity*.

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/ligandmpnn_designs.csv` + `results/proteinmpnn_designs.csv` exist.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. We reuse
the `"binder"` confidence cutoffs (scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10) and add a **specificity gate**
on top (a protein–NA-specific check the shared module does not encode).

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])
SPEC_MARGIN = 1.5   # min specificity_score (dScore) to call a design motif-specific (tune per project)
print("specificity gate: specificity_score >=", SPEC_MARGIN, "(applied AFTER the shared layers)")

## Build `Design` (binder) objects from the pools

Map each pool row onto `fp.Design` with `design_type="binder"`. The confidence metrics drive the shared
layers: `scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency) and `solubility` (Layer 3
physics). We carry `seq_tool`, `specificity_score`, and `is_specific` in `extra` for the specificity
gate + the nb04 head-to-head. (Mock has no independent orthogonal predictor, so we run Layers 1+3 here;
on Colab add a second protein–NA modeler for Layer 2.)

In [ ]:
import os
import pandas as pd

# Regenerate the pools if a fresh session lost them (deterministic mock).
if not (os.path.exists("results/ligandmpnn_designs.csv") and os.path.exists("results/proteinmpnn_designs.csv")):
    import na_binder_tools as nbt
    NA_TYPE = "DNA"; MOTIF = nbt.normalize_motif("TGACGTCA", NA_TYPE); SCRAMBLED = nbt.scramble_motif(MOTIF, seed=0)
    bbs = nbt.scaffold_near_na(MOTIF, n=40, tool="mock", na_type=NA_TYPE)
    lig, pro = [], []
    for bb in bbs:
        lig += nbt.ligandmpnn_na(bb, MOTIF, n=2, tool="mock", na_type=NA_TYPE, seq_tool="ligandmpnn")
        pro += nbt.ligandmpnn_na(bb, MOTIF, n=2, tool="mock", na_type=NA_TYPE, seq_tool="proteinmpnn")
    for grp in (lig, pro):
        nbt.score_designs(grp, na=MOTIF, tool="mock"); nbt.add_specificity(grp, MOTIF, scrambled=SCRAMBLED, tool="mock")
    def _q(designs, p):
        pd.DataFrame([dict(design_id=d.design_id, paradigm=d.paradigm, seq_tool=d.seq_tool,
                           na_type=d.na_type, target_motif=d.target_motif, length=d.length, sequence=d.sequence,
                           plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                           dG_motif=d.dG_motif, dG_scrambled=d.dG_scrambled,
                           specificity_score=d.specificity_score, is_specific=d.is_specific,
                           solubility=0.3, synthetic=d.synthetic) for d in designs]).to_csv(p, index=False)
    _q(lig, "results/ligandmpnn_designs.csv"); _q(pro, "results/proteinmpnn_designs.csv")

df_lig = pd.read_csv("results/ligandmpnn_designs.csv")
df_pro = pd.read_csv("results/proteinmpnn_designs.csv")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd modeler on Colab
        solubility=r.get("solubility", 0.3),
        # NOTE: the shared "binder" cutoffs include rosetta_dG/sc; for protein-NA we leave those None
        # (they apply to protein-protein interfaces) and gate on SPECIFICITY instead, below.
        extra={"seq_tool": r.get("seq_tool"),
               "specificity_score": r.get("specificity_score"),
               "is_specific": bool(r.get("is_specific"))},
    )

binders_lig = [row_to_binder(r) for _, r in df_lig.iterrows()]
binders_pro = [row_to_binder(r) for _, r in df_pro.iterrows()]
print(f"built {len(binders_lig)} LigandMPNN + {len(binders_pro)} ProteinMPNN binder Designs")

## Run the pipeline — per sequence designer (fair head-to-head)

`run_pipeline(design_type="binder")` applies the shared confidence cutoffs in order and returns a
ranked DataFrame with survival counts in `df.attrs`. We run **each designer separately** so the
survival funnels are comparable. We use Layers 1+3 here (mock has no independent orthogonal source; add
Layer 2 on Colab with a second protein–NA modeler). The protein–NA `"binder"` physics cutoffs for
`rosetta_dG`/`sc` are left unset (those are protein–protein metrics); **specificity is gated next**.

In [ ]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["seq_tool"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, confidence hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_lig = run_one(binders_lig, "ligandmpnn")
ranked_pro = run_one(binders_pro, "proteinmpnn")

ranked = pd.concat([ranked_lig, ranked_pro], ignore_index=True)
# Pull specificity out of `extra` into columns for the gate + nb04.
ranked["specificity_score"] = ranked["extra"].apply(lambda e: (e or {}).get("specificity_score"))
ranked["is_specific"] = ranked["extra"].apply(lambda e: bool((e or {}).get("is_specific")))
# Carry target_motif / na_type from the pools so they flow into top_candidates.csv (for the nb05 plan).
_meta = pd.concat([df_lig, df_pro], ignore_index=True).set_index("design_id")
for col in ("target_motif", "na_type"):
    ranked[col] = ranked["design_id"].map(_meta[col])
ranked = ranked.sort_values(["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "seq_tool", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "specificity_score", "is_specific"]]

## The specificity gate (protein–NA-specific, on top of the shared layers)

Confidence layers say the model is *sure where* the protein sits; they do **not** say it reads your
motif. The specificity gate keeps only designs that (a) pass the shared confidence layers **and** (b)
prefer the intended motif over the scrambled one (`specificity_score ≥ margin`). This funnel — confident
**and** specific — is the real protein–NA hit set. Expect a steep drop here: most confident designs are
non-specific backbone-grippers.

In [ ]:
conf = ranked[ranked["layers_passed"] >= 3]
spec = conf[conf["specificity_score"] >= SPEC_MARGIN]
print(f"confident (passed shared layers): {len(conf)}/{len(ranked)}")
print(f"confident AND motif-specific    : {len(spec)}/{len(ranked)}  "
      f"({100*len(spec)/max(len(ranked),1):.1f}%)   [SYNTHETIC if mock]")
for label, g in spec.groupby("seq_tool"):
    print(f"   {label:12s}: {len(g)} confident+specific designs")
spec.to_csv("results/specific_candidates.csv", index=False)
print("wrote results/specific_candidates.csv (the real protein-NA hit set)")

## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **combined**
pool for one comparable figure; the per-designer runs above are the rigorous version. Read the bars as
a funnel: steep drops show which layer discriminates (and remember the **specificity gate** above is the
protein–NA-specific drop the shared report does not draw).

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_lig + binders_pro
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p23")
print("\nsaved results/p23_survival.png + results/p23_ranked.csv")
top

## Honest hit-rate accounting (per designer, confidence vs specificity)

Report `N passing confidence layers / N generated` **and** `N confident-AND-specific / N generated`,
for **each** designer — these are the numbers the nb04 head-to-head builds on. The gap between them is
the specificity problem, quantified. Remember: survival is *enrichment*, not *correctness*. Mock numbers
are SYNTHETIC.

In [ ]:
for label, df in [("ligandmpnn", ranked_lig), ("proteinmpnn", ranked_pro)]:
    n = len(df); conf_n = int((df["layers_passed"] >= 3).sum())
    spec_n = int(spec[spec["seq_tool"] == label].shape[0])
    print(f"{label:12s}: confident = {conf_n}/{n} ({100*conf_n/max(n,1):.1f}%)  |  "
          f"confident+specific = {spec_n}/{n} ({100*spec_n/max(n,1):.1f}%)   [SYNTHETIC if mock]")
print("\nThe confident->specific drop IS the protein-NA challenge. Report both numbers, not just the best design.")

## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per sequence designer** (funnel figure `results/p23_survival.png`).
- [ ] **Specificity gate** applied on top (`results/specific_candidates.csv`): confident **AND** prefers the intended motif.
- [ ] Honest hit-rate accounting: confidence hit rate **and** confident-AND-specific rate for LigandMPNN and ProteinMPNN.
- [ ] Mapping assumptions (which fields → which `Design` attributes; why rosetta_dG/sc left unset for protein–NA) written down.

**Next:** `04_validate.ipynb` — LigandMPNN-vs-ProteinMPNN benchmark + specificity figures + complex modeling.